# Notebook 01 — Análisis Exploratorio de Datos (EDA)

**Objetivo:** Explorar el dataset *Cell Images for Detecting Malaria* antes de cualquier entrenamiento.

## Cómo ejecutar

### Local
```bash
# Desde la raíz del repo:
jupyter lab notebooks/01_eda.ipynb
```

### Google Colab
1. Sube este notebook a Colab (File → Upload notebook)
2. Descomenta las celdas marcadas con `# COLAB`
3. Ejecuta Runtime → Run all

In [ ]:
# ════════════════════════════════════════════════════════════════
# SETUP — Local o Colab (auto-detección)
# Pre-requisito Colab: Secrets GITHUB_TOKEN, GITHUB_USER, GITHUB_EMAIL
#                       (Herramientas → Secrets en el panel izquierdo)
# ════════════════════════════════════════════════════════════════
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata, drive

    GH_TOKEN = userdata.get("GITHUB_TOKEN")
    GH_USER  = userdata.get("GITHUB_USER")
    GH_EMAIL = userdata.get("GITHUB_EMAIL")
    REPO     = "Malaria-Dectetion-Deeplearning"
    REPO_URL = f"https://{GH_TOKEN}@github.com/{GH_USER}/{REPO}.git"

    if not os.path.exists(f"/content/{REPO}"):
        get_ipython().system(f"git clone -q {REPO_URL}")
    get_ipython().run_line_magic("cd", f"/content/{REPO}")
    get_ipython().system(f'git config user.email "{GH_EMAIL}"')
    get_ipython().system(f'git config user.name  "{GH_USER}"')
    get_ipython().system("git pull -q origin main")
    get_ipython().system("pip install -r requirements.txt -q")

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/malaria_project"
    get_ipython().system(f"mkdir -p {DRIVE_ROOT}/checkpoints {DRIVE_ROOT}/embeddings")
    # Solo lo pesado (no commit) va a Drive vía simlink
    get_ipython().system("rm -rf artifacts/checkpoints data/embeddings")
    get_ipython().system("mkdir -p artifacts data")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/checkpoints artifacts/checkpoints")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/embeddings   data/embeddings")
    # Lo ligero (sí commit) vive dentro del repo
    get_ipython().system("mkdir -p artifacts/figures artifacts/metrics artifacts/logs data/processed")
    print("✓ Colab listo. Pesados → Drive, ligeros → repo.")

# Detectar la raíz del repo (funciona en local y en Colab tras %cd)
cwd = Path().resolve()
REPO_ROOT = cwd if (cwd / "src").exists() else cwd.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"Working dir: {REPO_ROOT}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# DATASET — Kaggle API (solo si falta cell_images/)
# Pre-requisito: kaggle.json (Kaggle → Settings → Create API Token)
# Se cachea en Drive para no resubirlo en sesiones futuras
# ════════════════════════════════════════════════════════════════
if IN_COLAB and not os.path.exists("cell_images"):
    if os.path.exists(f"{DRIVE_ROOT}/kaggle.json"):
        get_ipython().system("mkdir -p ~/.kaggle")
        get_ipython().system(f"cp {DRIVE_ROOT}/kaggle.json ~/.kaggle/")
    else:
        from google.colab import files
        print("Sube tu kaggle.json (Kaggle → Settings → Create API Token):")
        files.upload()
        get_ipython().system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/")
        get_ipython().system(f"cp ~/.kaggle/kaggle.json {DRIVE_ROOT}/kaggle.json")
    get_ipython().system("chmod 600 ~/.kaggle/kaggle.json")
    get_ipython().system("pip install kaggle -q")
    get_ipython().system("kaggle datasets download -d iarunava/cell-images-for-detecting-malaria -q")
    get_ipython().system("unzip -q cell-images-for-detecting-malaria.zip")
    get_ipython().system("rm -f cell-images-for-detecting-malaria.zip")
    # El zip a veces anida cell_images/cell_images — aplanar:
    get_ipython().system("if [ -d cell_images/cell_images ]; then mv cell_images/cell_images/* cell_images/ 2>/dev/null; rmdir cell_images/cell_images 2>/dev/null; fi")
    print(f"✓ Dataset listo: {len(os.listdir('cell_images/Parasitized'))} parasitized, "
          f"{len(os.listdir('cell_images/Uninfected'))} uninfected")

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image

from src.utils.seed import set_global_seed
from src.utils.io import load_config
from src.data.split import make_stratified_split
from src.data.augmentations import get_contrastive_transform
from src.visualization.eda_plots import (
    plot_class_examples, plot_size_distribution,
    plot_intensity_histograms, plot_augmentation_examples
)

set_global_seed(42)
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
cfg = load_config('configs/data.yaml')
print('Config cargada:', cfg)

## 1. Generación del split estratificado

In [ ]:
train_df, val_df, test_df = make_stratified_split(
    dataset_root=cfg['dataset_root'],
    processed_dir=cfg['processed_dir'],
    train_frac=cfg['split']['train'],
    val_frac=cfg['split']['val'],
    seed=cfg['seed'],
    class_map=cfg['classes'],
)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\nDistribución Train:')
print(train_df['class_name'].value_counts())

## 2. Conteo y balance por clase

In [ ]:
full_df = pd.concat([train_df, val_df, test_df])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Conteo total
counts = full_df['class_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=['steelblue', 'tomato'], alpha=0.8)
axes[0].set_title('Distribución de clases — Dataset completo')
axes[0].set_ylabel('Número de imágenes')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Por split
splits = {'Train': train_df, 'Val': val_df, 'Test': test_df}
split_names = list(splits.keys())
for j, (split_name, df) in enumerate(splits.items()):
    cnt = df['class_name'].value_counts()
    axes[1].bar([j - 0.2, j + 0.2], cnt.values, 0.4,
                color=['steelblue', 'tomato'], alpha=0.8,
                label=['Uninfected', 'Parasitized'] if j == 0 else ['', ''])
axes[1].set_xticks(range(len(splits)))
axes[1].set_xticklabels(split_names)
axes[1].set_title('Distribución por split')
axes[1].set_ylabel('Número de imágenes')
axes[1].legend()

plt.tight_layout()
plt.savefig('artifacts/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ejemplos visuales por clase

In [ ]:
fig = plot_class_examples(full_df, n_per_class=8, seed=42,
                           save_path='artifacts/figures/class_examples.png')
plt.show()

## 4. Distribución de tamaños de imagen

In [ ]:
# Muestrear 500 imágenes para el análisis de tamaño
sample_df = full_df.sample(min(500, len(full_df)), random_state=42)
fig = plot_size_distribution(sample_df, save_path='artifacts/figures/size_distribution.png')
plt.show()

## 5. Histogramas de intensidad por canal

In [ ]:
fig = plot_intensity_histograms(full_df, n_samples=100, seed=42,
                                 save_path='artifacts/figures/intensity_histograms.png')
plt.show()

## 6. Análisis de brillo y contraste

In [ ]:
from PIL import ImageStat

brightness_per_class = {'Parasitized': [], 'Uninfected': []}
contrast_per_class = {'Parasitized': [], 'Uninfected': []}

sample = full_df.sample(min(300, len(full_df)), random_state=42)
for _, row in sample.iterrows():
    try:
        img = Image.open(row['path']).convert('RGB')
        stat = ImageStat.Stat(img)
        brightness = np.mean(stat.mean)
        contrast = np.mean(stat.stddev)
        brightness_per_class[row['class_name']].append(brightness)
        contrast_per_class[row['class_name']].append(contrast)
    except Exception:
        pass

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cls, color in [('Parasitized', 'tomato'), ('Uninfected', 'steelblue')]:
    axes[0].hist(brightness_per_class[cls], bins=30, alpha=0.6, label=cls, color=color)
    axes[1].hist(contrast_per_class[cls], bins=30, alpha=0.6, label=cls, color=color)

axes[0].set_title('Distribución de brillo')
axes[0].set_xlabel('Brillo medio')
axes[0].legend()
axes[1].set_title('Distribución de contraste')
axes[1].set_xlabel('Desviación estándar de píxeles')
axes[1].legend()
plt.tight_layout()
plt.savefig('artifacts/figures/brightness_contrast.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Detección de archivos corruptos

In [ ]:
from PIL import UnidentifiedImageError

corrupted = []
for path in full_df['path']:
    try:
        with Image.open(path) as img:
            img.verify()
    except (UnidentifiedImageError, Exception):
        corrupted.append(path)

print(f'Total imágenes: {len(full_df)}')
print(f'Corruptas: {len(corrupted)}')
if corrupted:
    print('Archivos corruptos:')
    for p in corrupted:
        print(f'  {p}')

## 8. Ejemplos de augmentations contrastivas

In [ ]:
sample_path = train_df[train_df['class_name'] == 'Parasitized']['path'].iloc[0]
transform = get_contrastive_transform(img_size=96)
fig = plot_augmentation_examples(sample_path, transform, n_augmentations=6,
                                  save_path='artifacts/figures/augmentation_examples.png')
plt.show()
print(f'\nEjemplo de: {sample_path}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# SYNC CON GITHUB — push del notebook ejecutado + artefactos ligeros
# Pesados (.pt/.npy) ya están en Drive y excluidos por .gitignore
# ════════════════════════════════════════════════════════════════
if IN_COLAB:
    NOTEBOOK = "01_eda"
    # Forzar guardado del .ipynb (para preservar outputs)
    try:
        from google.colab import _message
        _message.blocking_request("save_notebook", request="", timeout_sec=10)
    except Exception:
        pass
    get_ipython().system(
        "git add notebooks/{nb}.ipynb artifacts/figures artifacts/metrics artifacts/logs data/processed".format(nb=NOTEBOOK)
    )
    get_ipython().system(f'git commit -m "nb {NOTEBOOK}: ejecutado en Colab con outputs" || echo "Sin cambios para commitear"')
    get_ipython().system("git push -q origin main && echo '✓ Pushed a GitHub' || echo '⚠ Push falló (revisa GITHUB_TOKEN)'")

## Resumen EDA

| Métrica | Valor |
|---|---|
| Total imágenes | 27,558 |
| Parasitized | 13,779 (50%) |
| Uninfected | 13,779 (50%) |
| Balance | Perfecto (no requiere oversampling) |
| Train | ~19,290 imágenes |
| Val | ~4,134 imágenes |
| Test | ~4,134 imágenes |
